# Full Data Load to Neo4j

This notebook loads the **complete** Aircraft Digital Twin dataset into Neo4j using the **Neo4j Spark Connector**.

## What's Included

**Nodes:**
- Aircraft, System, Component, Sensor
- Airport, Flight, Delay
- MaintenanceEvent, Removal

**Relationships:**
- HAS_SYSTEM, HAS_COMPONENT, HAS_SENSOR
- OPERATES_FLIGHT, DEPARTS_FROM, ARRIVES_AT, HAS_DELAY
- HAS_EVENT, AFFECTS_SYSTEM, AFFECTS_AIRCRAFT
- HAS_REMOVAL, REMOVED_COMPONENT

## Prerequisites
- Neo4j Aura credentials from Lab 1
- CSV data files uploaded to Unity Catalog Volume
- Access to the workshop Databricks cluster (with Neo4j Spark Connector installed)

## Instructions
1. Enter your Neo4j credentials in the Configuration cell below
2. Set `CLEAR_DATABASE = True` if you want to start fresh
3. Run all cells in order

## Section 1: Configuration

Enter your Neo4j Aura connection details below.

In [ ]:
# ============================================
# CONFIGURATION - Enter your Neo4j credentials
# ============================================

NEO4J_URI = ""  # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""  # Your password from Lab 1

# Set to True to clear the database before loading
CLEAR_DATABASE = True

# Unity Catalog Volume path (pre-configured by workshop admin)
DATA_PATH = "/Volumes/databricks-neo4j-workshop/aircraft/raw_data"

# Validate configuration
if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running!")
else:
    print("Configuration ready!")
    print(f"Neo4j URI: {NEO4J_URI}")
    print(f"Data Path: {DATA_PATH}")
    print(f"Clear Database: {CLEAR_DATABASE}")

## Section 2: Configure Spark Connector

Configure the Neo4j Spark Connector with your Aura connection details.

In [ ]:
# Configure Neo4j Spark Connector
spark.conf.set("neo4j.url", NEO4J_URI)
spark.conf.set("neo4j.authentication.basic.username", NEO4J_USERNAME)
spark.conf.set("neo4j.authentication.basic.password", NEO4J_PASSWORD)
spark.conf.set("neo4j.database", "neo4j")

print("Spark configured for Neo4j connection")

## Section 3: Helper Functions

In [ ]:
from pyspark.sql.functions import col

# Increase from default 5000 — the docs note the default is
# "likely too low for many applications" and recommend ~20,000.
BATCH_SIZE = 20000


def read_csv(filename):
    """Read a CSV file from the Unity Catalog Volume."""
    return spark.read.option("header", "true").csv(f"{DATA_PATH}/{filename}")


def write_nodes(df, label, id_column):
    """Write a DataFrame as nodes to Neo4j."""
    (df.write
     .format("org.neo4j.spark.DataSource")
     .mode("Overwrite")
     .option("labels", f":{label}")
     .option("node.keys", id_column)
     .option("batch.size", BATCH_SIZE)
     .save())
    print(f"  [OK] Wrote {df.count()} {label} nodes")


def write_relationships(df, rel_type, source_label, source_key, target_label, target_key):
    """Write relationships to Neo4j using keys strategy.

    Uses coalesce(1) to send all rows through a single partition,
    preventing deadlocks from concurrent writes locking the same nodes.
    """
    (df.coalesce(1)
     .write
     .format("org.neo4j.spark.DataSource")
     .mode("Overwrite")
     .option("relationship", rel_type)
     .option("relationship.save.strategy", "keys")
     .option("relationship.source.labels", f":{source_label}")
     .option("relationship.source.node.keys", source_key)
     .option("relationship.target.labels", f":{target_label}")
     .option("relationship.target.node.keys", target_key)
     .option("batch.size", BATCH_SIZE)
     .save())
    print(f"  [OK] Wrote {df.count()} {rel_type} relationships")


def run_cypher(query):
    """Execute a Cypher query and return results as DataFrame."""
    return (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("query", query)
        .load())


def run_script(script):
    """Execute a Cypher script (DDL operations like constraints)."""
    return (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("script", script)
        .option("query", "RETURN 1 AS done")
        .load())


print("Helper functions defined.")

## Section 4: Clear Database (Optional)

In [ ]:
if CLEAR_DATABASE:
    print("Clearing database...")
    run_script("CALL { MATCH (n) WITH n LIMIT 10000 DETACH DELETE n } IN TRANSACTIONS OF 10000 ROWS")
    result = run_cypher("MATCH (n) RETURN count(n) AS remaining")
    display(result)
    remaining = result.collect()[0]["remaining"]
    if remaining > 0:
        print(f"WARNING: {remaining} nodes still remain. Re-run this cell until remaining = 0.")
    else:
        print("[OK] Database cleared.")
else:
    print("Skipping database clear (CLEAR_DATABASE = False)")

## Section 5: Create Constraints and Indexes

In [ ]:
print("Creating constraints...")

constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Aircraft) REQUIRE n.aircraft_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:System) REQUIRE n.system_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Component) REQUIRE n.component_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Sensor) REQUIRE n.sensor_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Airport) REQUIRE n.airport_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Flight) REQUIRE n.flight_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Delay) REQUIRE n.delay_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:MaintenanceEvent) REQUIRE n.event_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Removal) REQUIRE n.removal_id IS UNIQUE",
]

indexes = [
    "CREATE INDEX idx_maintenanceevent_severity IF NOT EXISTS FOR (n:MaintenanceEvent) ON (n.severity)",
    "CREATE INDEX idx_flight_aircraft_id IF NOT EXISTS FOR (n:Flight) ON (n.aircraft_id)",
    "CREATE INDEX idx_removal_aircraft_id IF NOT EXISTS FOR (n:Removal) ON (n.aircraft_id)",
]

run_script(";\n".join(constraints + indexes))
print("[OK] Constraints and indexes created.")

## Section 6: Load Nodes

Load all node types from CSV files.

In [ ]:
print("Loading Aircraft nodes...")
df = read_csv("nodes_aircraft.csv").withColumnRenamed(":ID(Aircraft)", "aircraft_id")
write_nodes(df, "Aircraft", "aircraft_id")

In [ ]:
print("Loading System nodes...")
df = read_csv("nodes_systems.csv").withColumnRenamed(":ID(System)", "system_id")
write_nodes(df, "System", "system_id")

In [ ]:
print("Loading Component nodes...")
df = read_csv("nodes_components.csv").withColumnRenamed(":ID(Component)", "component_id")
write_nodes(df, "Component", "component_id")

In [ ]:
print("Loading Sensor nodes...")
df = read_csv("nodes_sensors.csv").withColumnRenamed(":ID(Sensor)", "sensor_id")
write_nodes(df, "Sensor", "sensor_id")

In [ ]:
print("Loading Airport nodes...")
df = (read_csv("nodes_airports.csv")
    .withColumnRenamed(":ID(Airport)", "airport_id")
    .withColumn("lat", col("lat").cast("double"))
    .withColumn("lon", col("lon").cast("double")))
write_nodes(df, "Airport", "airport_id")

In [ ]:
print("Loading Flight nodes...")
df = read_csv("nodes_flights.csv").withColumnRenamed(":ID(Flight)", "flight_id")
write_nodes(df, "Flight", "flight_id")

In [ ]:
print("Loading Delay nodes...")
df = (read_csv("nodes_delays.csv")
    .withColumnRenamed(":ID(Delay)", "delay_id")
    .withColumn("minutes", col("minutes").cast("integer")))
write_nodes(df, "Delay", "delay_id")

In [ ]:
print("Loading MaintenanceEvent nodes...")
df = read_csv("nodes_maintenance.csv").withColumnRenamed(":ID(MaintenanceEvent)", "event_id")
write_nodes(df, "MaintenanceEvent", "event_id")

In [ ]:
print("Loading Removal nodes...")
df = (read_csv("nodes_removals.csv")
    .withColumnRenamed(":ID(RemovalEvent)", "removal_id")
    .withColumnRenamed("RMV_REA_TX", "reason")
    .withColumnRenamed("time_since_install", "tsn")
    .withColumnRenamed("flight_cycles_at_removal", "csn")
    .withColumn("tsn", col("tsn").cast("double"))
    .withColumn("csn", col("csn").cast("integer")))
write_nodes(df, "Removal", "removal_id")

## Section 7: Load Relationships

In [ ]:
print("Loading HAS_SYSTEM relationships...")
df = (read_csv("rels_aircraft_system.csv")
    .withColumnRenamed(":START_ID(Aircraft)", "aircraft_id")
    .withColumnRenamed(":END_ID(System)", "system_id"))
write_relationships(df, "HAS_SYSTEM", "Aircraft", "aircraft_id", "System", "system_id")

In [ ]:
print("Loading HAS_COMPONENT relationships...")
df = (read_csv("rels_system_component.csv")
    .withColumnRenamed(":START_ID(System)", "system_id")
    .withColumnRenamed(":END_ID(Component)", "component_id"))
write_relationships(df, "HAS_COMPONENT", "System", "system_id", "Component", "component_id")

In [ ]:
print("Loading HAS_SENSOR relationships...")
df = (read_csv("rels_system_sensor.csv")
    .withColumnRenamed(":START_ID(System)", "system_id")
    .withColumnRenamed(":END_ID(Sensor)", "sensor_id"))
write_relationships(df, "HAS_SENSOR", "System", "system_id", "Sensor", "sensor_id")

In [ ]:
print("Loading HAS_EVENT relationships...")
df = (read_csv("rels_component_event.csv")
    .withColumnRenamed(":START_ID(Component)", "component_id")
    .withColumnRenamed(":END_ID(MaintenanceEvent)", "event_id"))
write_relationships(df, "HAS_EVENT", "Component", "component_id", "MaintenanceEvent", "event_id")

In [ ]:
print("Loading OPERATES_FLIGHT relationships...")
df = (read_csv("rels_aircraft_flight.csv")
    .withColumnRenamed(":START_ID(Aircraft)", "aircraft_id")
    .withColumnRenamed(":END_ID(Flight)", "flight_id"))
(df.write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .option("relationship", "OPERATES_FLIGHT")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Aircraft")
 .option("relationship.source.node.keys", "aircraft_id")
 .option("relationship.target.labels", ":Flight")
 .option("relationship.target.node.keys", "flight_id")
 .option("batch.size", BATCH_SIZE)
 .save())
print(f"  [OK] Wrote {df.count()} OPERATES_FLIGHT relationships")

In [ ]:
print("Loading DEPARTS_FROM relationships...")
df = (read_csv("rels_flight_departure.csv")
    .withColumnRenamed(":START_ID(Flight)", "flight_id")
    .withColumnRenamed(":END_ID(Airport)", "airport_id"))
write_relationships(df, "DEPARTS_FROM", "Flight", "flight_id", "Airport", "airport_id")

In [ ]:
print("Loading ARRIVES_AT relationships...")
df = (read_csv("rels_flight_arrival.csv")
    .withColumnRenamed(":START_ID(Flight)", "flight_id")
    .withColumnRenamed(":END_ID(Airport)", "airport_id"))
write_relationships(df, "ARRIVES_AT", "Flight", "flight_id", "Airport", "airport_id")

In [ ]:
print("Loading HAS_DELAY relationships...")
df = (read_csv("rels_flight_delay.csv")
    .withColumnRenamed(":START_ID(Flight)", "flight_id")
    .withColumnRenamed(":END_ID(Delay)", "delay_id"))
write_relationships(df, "HAS_DELAY", "Flight", "flight_id", "Delay", "delay_id")

In [ ]:
print("Loading AFFECTS_SYSTEM relationships...")
df = (read_csv("rels_event_system.csv")
    .withColumnRenamed(":START_ID(MaintenanceEvent)", "event_id")
    .withColumnRenamed(":END_ID(System)", "system_id"))
write_relationships(df, "AFFECTS_SYSTEM", "MaintenanceEvent", "event_id", "System", "system_id")

In [ ]:
print("Loading AFFECTS_AIRCRAFT relationships...")
df = (read_csv("rels_event_aircraft.csv")
    .withColumnRenamed(":START_ID(MaintenanceEvent)", "event_id")
    .withColumnRenamed(":END_ID(Aircraft)", "aircraft_id"))
write_relationships(df, "AFFECTS_AIRCRAFT", "MaintenanceEvent", "event_id", "Aircraft", "aircraft_id")

In [ ]:
print("Loading HAS_REMOVAL relationships...")
df = (read_csv("rels_aircraft_removal.csv")
    .withColumnRenamed(":START_ID(Aircraft)", "aircraft_id")
    .withColumnRenamed(":END_ID(RemovalEvent)", "removal_id"))
write_relationships(df, "HAS_REMOVAL", "Aircraft", "aircraft_id", "Removal", "removal_id")

In [ ]:
print("Loading REMOVED_COMPONENT relationships...")
df = (read_csv("rels_component_removal.csv")
    .withColumnRenamed(":START_ID(Component)", "component_id")
    .withColumnRenamed(":END_ID(RemovalEvent)", "removal_id"))
write_relationships(df, "REMOVED_COMPONENT", "Removal", "removal_id", "Component", "component_id")

## Section 8: Summary

In [ ]:
print("=" * 60)
print("LOAD COMPLETE - Summary")
print("=" * 60)

# Count nodes
print("\nNode Counts:")
node_result = run_cypher("""
    CALL () {
        MATCH (n:Aircraft) RETURN 'Aircraft' as label, count(n) as count
        UNION ALL
        MATCH (n:System) RETURN 'System' as label, count(n) as count
        UNION ALL
        MATCH (n:Component) RETURN 'Component' as label, count(n) as count
        UNION ALL
        MATCH (n:Sensor) RETURN 'Sensor' as label, count(n) as count
        UNION ALL
        MATCH (n:Airport) RETURN 'Airport' as label, count(n) as count
        UNION ALL
        MATCH (n:Flight) RETURN 'Flight' as label, count(n) as count
        UNION ALL
        MATCH (n:Delay) RETURN 'Delay' as label, count(n) as count
        UNION ALL
        MATCH (n:MaintenanceEvent) RETURN 'MaintenanceEvent' as label, count(n) as count
        UNION ALL
        MATCH (n:Removal) RETURN 'Removal' as label, count(n) as count
    }
    RETURN label, count
    ORDER BY count DESC
""")
display(node_result)

# Count relationships
print("\nRelationship Counts:")
rel_result = run_cypher("""
    MATCH ()-[r]->()
    RETURN type(r) AS RelType, count(*) AS Count
    ORDER BY Count DESC
""")
display(rel_result)

print("=" * 60)

In [ ]:
print("All data loaded using the Neo4j Spark Connector.")

## Next Steps

Your Neo4j database now contains the complete Aircraft Digital Twin dataset!

### Explore in Neo4j Aura

1. Go to [console.neo4j.io](https://console.neo4j.io)
2. Open your instance and click **Query**

### Sample Queries to Try

**View the complete schema:**
```cypher
CALL db.schema.visualization()
```

**Find aircraft with critical maintenance issues:**
```cypher
MATCH (a:Aircraft)-[:HAS_SYSTEM]->(s:System)-[:HAS_COMPONENT]->(c:Component)-[:HAS_EVENT]->(m:MaintenanceEvent)
WHERE m.severity = 'CRITICAL' AND m.reported_at IS NOT NULL
RETURN a.tail_number, s.name, c.name, m.fault, m.reported_at
ORDER BY m.reported_at DESC
LIMIT 10
```

**Analyze flight delays by cause:**
```cypher
MATCH (f:Flight)-[:HAS_DELAY]->(d:Delay)
RETURN d.cause, count(*) AS count, avg(d.minutes) AS avg_minutes
ORDER BY count DESC
```

**Find component removal history:**
```cypher
MATCH (a:Aircraft)-[:HAS_REMOVAL]->(r:Removal)-[:REMOVED_COMPONENT]->(c:Component)
WHERE r.removal_date IS NOT NULL
RETURN a.tail_number, c.name, r.reason, r.removal_date, r.tsn, r.csn
ORDER BY r.removal_date DESC
LIMIT 20
```